In [0]:
%python
from pyspark.sql.functions import *

In [0]:
%python
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

# Define the schema
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("contact_number", StringType(), True),
    StructField("mail_id", StringType(), True),
    StructField("status", StringType(), True),
])

# Define the data rows
data = [
    (1, "+91 9876543210", "john.doe@gmail.com", "Active"),
    (2, "+1 9123456780", "jane.smith@gmail.com", "Inactive"),
    (3, "+44 9988776655", "mike.brown@gmail.com", "Active"),
    (4, "+91 9012345678", "emily.white@gmail.com", "Inactive"),
    (5, "+61 9090909090", "robert.black@gmail.com", "Active"),
    (6, "+1 8887776665", "sara.green@gmail.com", "Active"),
]

# Create the DataFrame
employee_df = spark.createDataFrame(data=data, schema=schema)

# Display the DataFrame
employee_df.show(truncate=False)


Mask the contact number except the last 4 didgits

In [0]:
%python
from pyspark.sql import functions as F

# Mask the 6 digits following the country code and space, keeping the last 4 digits
masked_df = employee_df.withColumn(
    "contact_number",
    F.regexp_replace("contact_number", r"^(\+\d+\s)\d{6}(\d{4})$", r"$1******$2"),
)

masked_df.show(truncate=False)

# How it works:

# • ^(\+\d+\s): Captures the +, the country code digits (\d+), and the space (\s) as group $1.
# • \d{6}: Matches the next 6 digits to be replaced with ******.
# • (\d{4})$: Captures the final 4 digits as group $2.

In [0]:
-- Regular Expressions (Regex) are sequences of characters that define a search pattern, commonly used in PySpark and SQL to match, extract, or replace text.
-- Here are the core building blocks you need to read and write regex patterns like the one used above.

-- 1. Character Classes (What to match)

-- These symbols act as wildcards for specific types of characters:

-- Symbol	Matches	Example
-- \d	Any digit (0-9)	\d\d matches 42
-- \D	Any non-digit	\D matches A or @
-- \w	Any word character (letters, digits, underscore)	\w+ matches user_1
-- \s	Any whitespace (space, tab, newline)	a\sb matches a b
-- .	Any single character (except newline)	c.t matches cat, cot, c9t
-- [abc]	Any single character inside the brackets	[aeiou] matches any lowercase vowel


-- 2. Quantifiers (How many times to match)

-- Quantifiers tell the engine how many repetitions of the preceding character to look for:

-- Symbol	Meaning	Example
-- +	1 or more times	\d+ matches 1, 91, or 98765
-- *	0 or more times	ab* matches a, ab, abbb
-- ?	0 or 1 time (makes it optional)	colou?r matches color and colour
-- {n}	Exactly n times	\d{4} matches 3210
-- {n,m}	Between n and m times	\d{2,4} matches 12, 123, or 1234


-- 3. Anchors & Escaping (Where to match)

-- • ^ (Caret): Matches the start of a string. (^\+ ensures + is the very first character).
-- • $ (Dollar): Matches the end of a string. (\d{4}$ ensures the 4 digits are at the very end).
-- • \ (Backslash): Escapes special characters. Because +, ., and ? have special regex meanings, you must put a \ in front of them to match them literally (e.g., \+ matches a literal plus sign).

-- 4. Capture Groups (Saving parts of a match)

-- Wrapping part of a pattern in parentheses () saves whatever text it matches into memory so you can reuse it later:
-- • The first (...) is stored as $1 (or \1 in some languages).
-- • The second (...) is stored as $2.

-- Breaking Down Your Previous Pattern

-- Here is how those rules combined to mask +91 9876543210 into +91 ******3210:
-- ^(\+\d+\s)\d{6}(\d{4})$
-- 1. ^ — Start at the beginning of the string.
-- 2. (\+\d+\s) — Group $1: Match a literal +, followed by 1 or more digits (91), followed by a space. Save this.
-- 3. \d{6} — Match the next 6 digits (987654), do not save them.
-- 4. (\d{4}) — Group $2: Match the next 4 digits (3210). Save this.
-- 5. $ — Ensure we reached the end of the string.
-- When replaced with $1******$2, Spark drops the ungrouped middle digits, puts six asterisks in their place, and stitches Group 1 and Group 2 back together.
-- If you'd like to practice, I can:
-- • Show the other main PySpark regex functions (regexp_extract, rlike)
-- • Walk through a pattern for masking or validating the mail_id column
-- Let me know what you want to explore next.


In [0]:
%python
(employee_df.withColumn("split_cont",split(col("contact_number"), " "))
            #.withColumn("masked_num1",right(col("split_cont")[1], lit(4))).show())
            .withColumn("masked_num1",regexp_replace(col("split_cont")[1],r"(^\d{6})","******")).show(truncate=False))
            
                        # regexp_replace(col("split_cont")[1],"(^\d{6})","******")).show(truncate=False)
           

In [0]:
%python
# employee_df.withColumn("split_cont",split(col("contact_number"), " ")).withColumn("split_mail",split(col("mail_id"), "@"))
(employee_df.withColumn("split_mail",split(col("mail_id"), "@")[0]) 
 .withColumn("split_cont",split(col("split_mail"), r"\.")) # or .withColumn("split_cont",split(col("split_mail"), "\\."))
 .withColumn("first_name",col("split_cont")[0])
 .withColumn("last_name",col("split_cont")[1]).show())


# How to Fix It
# You must escape the dot using double backslashes (\\.) or a raw string with a single backslash (r"\.") so regex treats it as a literal period:

In [0]:
%python
(employee_df.withColumn("split_cont",split(col("contact_number"), " "))
            .withColumn("masked_num1",right(col("split_cont")[1], lit(4))).show())

In [0]:
%python
employee_df.withColumn("split_cont",split(col("contact_number"), " ")) \
           .withColumn("masked_num1", F.substring(F.col("split_cont")[1], -4, 4)).show()

In [0]:
%python
(employee_df.withColumn("split_cont",split(col("contact_number"), " ")).withColumn("masked_num1", concat(F.lit("******"),(F.substring(F.col("split_cont")[1], -4, 4)))).show())

In [0]:
%python
(employee_df.withColumn("split_cont",split(col("contact_number"), " ")).withColumn("masked_num1", concat(col("split_cont")[0],lit(" "),F.lit("******"),(F.substring(F.col("split_cont")[1], -4, 4)))).show())

Same thing in **SQL**

In [0]:
%python
employee_df.createOrReplaceTempView("employee")
       


In [0]:
select *
,split(contact_number,' ')[0] as prefix
,split(contact_number,' ')[1] as num
,right(num,4) as last4
,concat(prefix,' ',concat('*******',last4)) as masked1
,regexp_replace(contact_number,'[0-9]','*') as masked2
,concat(prefix,' ',concat(repeat('*',length(num)-4),last4)) as masked3
 from employee;